In [2]:
import geopandas as gpd

# Preparing for GLASGOW GWR 10x10m 

## Glasgow Converting 10m NDVI

In [ ]:
#C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/winter/gg_sum_10m_rast.tif

### reading the NDVI raster as polygons 

In [193]:
import geopandas as gpd
import rasterio
from rasterio.features import shapes
import numpy as np
from shapely.geometry import shape

with rasterio.open(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\Round3\sum_2022_groundcover.geojson") as src:
    noise_array = src.read(1)
    transform = src.transform

results = [
    {"type": "Feature",
     "geometry": shape(geom),
     "properties": {"value": value}}
    for geom, value in shapes(noise_array, transform=transform)
    if value is not None
]

ndvi = gpd.GeoDataFrame.from_features(results, crs=src.crs)

ndvi.rename(columns={"value": "G"}, inplace=True)

In [4]:
import geopandas as gpd

In [5]:
ndvi = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_ndvi_clipped.shp")

In [6]:
ndvi.head()

,fid,G,geometry
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67..."
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67..."
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67..."
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67..."
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67..."


### joining NDVI with landcover data 
took about ten mins

In [23]:
landcover = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Results\Landcoverc\gg_landcover.shp")
ndvi = gpd.sjoin(ndvi, landcover[["geometry", "lc_class"]], how="left", predicate="intersects")

In [29]:
gwr_n_lc = ndvi

In [30]:
gwr_n_lc.head()

,fid,G,geometry,mean_noise,index_right,lc_class
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67...",0.0,6964.0,2.0
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67...",0.0,6964.0,2.0
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67...",0.0,6941.0,2.0
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67...",0.0,6941.0,2.0
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67...",0.0,6964.0,2.0


In [32]:
gwr_n_lc["mean_noise"].describe()

count    244233.000000
mean          5.111969
std           2.312875
min           0.000000
25%           3.658386
50%           5.380785
75%           6.672368
max          10.000000
Name: mean_noise, dtype: float64

In [33]:
gwr_n_lc

,fid,G,geometry,mean_noise,index_right,lc_class
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67...",0.000000,6964.0,2.0
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67...",0.000000,6964.0,2.0
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67...",0.000000,6941.0,2.0
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67...",0.000000,6941.0,2.0
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67...",0.000000,6964.0,2.0
...,...,...,...,...,...,...
113912,113913.0,0.000000,"MULTIPOLYGON (((260155.601 660741.104, 260155....",2.920558,6964.0,2.0
113912,113913.0,0.000000,"MULTIPOLYGON (((260155.601 660741.104, 260155....",2.920558,2698.0,1.0
113913,113914.0,0.954866,"POLYGON ((260389.955 656497.068, 260389.955 65...",0.000000,6984.0,2.0
113914,113915.0,0.936278,"POLYGON ((260459.308 656488.846, 260429.275 65...",0.000000,6984.0,2.0


In [8]:
from rasterstats import zonal_stats
stats = zonal_stats(ndvi, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_10_rast_filled.tif", stats=["mean"], nodata=None)

In [11]:
ndvi["mean_noise"] = [s["mean"] for s in stats]

In [14]:
ndvi.head()

,fid,G,geometry,mean_noise
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67...",0.0
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67...",0.0
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67...",0.0
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67...",0.0
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67...",0.0


In [17]:
ndvi["mean_noise"].describe()

count    113916.000000
mean          4.574655
std           2.436981
min           0.000000
25%           3.000000
50%           4.836787
75%           6.269535
max          10.000000
Name: mean_noise, dtype: float64

## Trees

In [24]:
import geopandas as gpd
import pandas as pd
import glob
import os

# Folder containing the shapefiles
folder_path = r"C:\Users\ibk1\NoiseModelling\Glasgow\env_data\tree_volume\tree_canopy_products\treetop_location"

# Get a list of all shapefiles in the folder
shapefiles = glob.glob(os.path.join(folder_path, "*.shp"))

# Read and process each shapefile
gdfs = [gpd.read_file(shp).set_crs(epsg=27700, allow_override=True) for shp in shapefiles]

# Merge all shapefiles into a single GeoDataFrame
merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
clip_trees = gpd.clip(merged_gdf, ggbounds)

In [34]:
if "index_right" in gwr_n_lc.columns:
    gwr_n_lc = gwr_n_lc.drop(columns=["index_right"])

In [35]:
# Here is another way 
tree_counts = gpd.sjoin(clip_trees, gwr_n_lc, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
gwr_n_lc["tree_count"] = gwr_n_lc.index.map(tree_summary).fillna(0).astype(int)

In [36]:
gwr_n_lc.head()

,fid,G,geometry,mean_noise,lc_class,tree_count
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67...",0.0,2.0,7
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67...",0.0,2.0,14
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67...",0.0,2.0,1
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67...",0.0,2.0,7
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67...",0.0,2.0,11


In [40]:
gwr_n_lc["lc_class"].unique()

array(['2.0', '1.0', '0.0', '3.0', 'nan'], dtype=object)

In [53]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype(str)

# Step 2: Filter out rows where lc_class is '0.0' or 'nan'
gwr_n_lc = gwr_n_lc[~gwr_n_lc["lc_class"].isin(["0.0", "nan"])]

In [54]:
gwr_n_lc["lc_class"].unique()

array(['2.0', '1.0', '3.0'], dtype=object)

# Dummy coding 

In [55]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype("category")

C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [56]:
gwr_n_lc.head()

,fid,G,geometry,mean_noise,lc_class,tree_count
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67...",0.0,2.0,7
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67...",0.0,2.0,14
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67...",0.0,2.0,1
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67...",0.0,2.0,7
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67...",0.0,2.0,11


In [57]:
import pandas as pd

# Create dummy variables, drop_first=True avoids multicollinearity (reference category)
dummies = pd.get_dummies(gwr_n_lc["lc_class"], prefix="lc", drop_first=True)

# Join them back to the GeoDataFrame
gwr_n_lc_d = gwr_n_lc.join(dummies)

In [60]:
gwr_n_lc_d["tree_count"].describe()

count    670874.000000
mean      14126.691693
std       55394.537909
min           0.000000
25%           4.000000
50%          15.000000
75%          32.000000
max      245440.000000
Name: tree_count, dtype: float64

In [61]:
gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_10m.shp")

C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: 2GB file size limit reached for C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_10m.shp. Going on, but might cause compatibility issues with third party software
  ogr_write(


In [62]:
gwr_n_lc_d.info

<bound method DataFrame.info of              fid         G                                           geometry   
0            1.0  0.938139  POLYGON ((257873.463 673070.436, 257873.463 67...  \
1            2.0  0.935334  POLYGON ((257878.802 673071.405, 257893.198 67...   
2            3.0  0.802828  POLYGON ((256518.858 673042.643, 256536.577 67...   
3            4.0  0.739121  POLYGON ((256575.897 673026.291, 256575.897 67...   
4            5.0  0.943165  POLYGON ((257797.299 673005.604, 257794.823 67...   
...          ...       ...                                                ...   
113912  113913.0  0.000000  MULTIPOLYGON (((260155.601 660741.104, 260155....   
113912  113913.0  0.000000  MULTIPOLYGON (((260155.601 660741.104, 260155....   
113913  113914.0  0.954866  POLYGON ((260389.955 656497.068, 260389.955 65...   
113914  113915.0  0.936278  POLYGON ((260459.308 656488.846, 260429.275 65...   
113915  113916.0  0.916431  POLYGON ((260448.696 656462.599, 260429.275 65...

## centriods 

In [63]:
# Create a new GeoDataFrame with centroids
centroids_gwr_n_lc_d = gwr_n_lc_d.copy()
centroids_gwr_n_lc_d["geometry"] = gwr_n_lc_d.geometry.centroid

In [64]:
centroids_gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_10m_centriods.shp")

### Creating Urban pct test

In [250]:
noise_gdf["grid_id"] = noise_gdf.index.astype(str)

In [251]:
# testing 
urban_codes = [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]
urban = landcover[landcover["code_2018"].astype(int).isin(urban_codes)]

In [ ]:
urban_overlap = gpd.overlay(noise_gdf, urban, how="intersection")
urban_overlap["urban_area"] = urban_overlap.geometry.area

In [ ]:
urban_by_grid = (
    urban_overlap.groupby("grid_id")["urban_area"]
    .sum()
    .reset_index()
    .rename(columns={"urban_area": "urban_area_m2"})
)

In [ ]:
# Add to full grid
noise_gdf["grid_area"] = noise_gdf.geometry.area  # should be 100 m² for 10x10m
noise_gdf = noise_gdf.merge(urban_by_grid, on="grid_id", how="left")

# Fill NaNs (no urban overlap) with 0
noise_gdf["urban_area_m2"] = noise_gdf["urban_area_m2"].fillna(0)

# Calculate proportion
noise_gdf["urban_pct"] = noise_gdf["urban_area_m2"] / noise_gdf["grid_area"]


### joining with tree count data 

In [197]:
clip_trees.columns

Index(['Z', 'U_ID', 'geometry'], dtype='object')

In [198]:
noise_gdf.columns

Index(['geometry', 'noise', 'index_right', 'code_2018'], dtype='object')

In [199]:
if "index_right" in noise_gdf.columns:
    noise_gdf = noise_gdf.drop(columns=["index_right"])

In [244]:
tree_counts = gpd.sjoin(merged_gdf, noise_gdf, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
noise_gdf["tree_count"] = noise_gdf.index.map(tree_summary).fillna(0).astype(int)

KeyboardInterrupt: 

In [ ]:
tree_summary

In [ ]:
noise_gdf.head()

In [214]:
noise_gdf["code_2018"].unique()

array(['23000', '12220', '12100', '21000', '11300', nan, '32000', '31000',
       '14100', '11210', '11220', '14200', '50000', '13400', '13100',
       '13300', '11230', '11240', '12230', '11100', '12210', '12300'],
      dtype=object)

In [220]:
noise_gdf_cleaned = noise_gdf.dropna(subset=["code_2018"])

In [221]:
noise_gdf_cleaned.head()

,geometry,noise,code_2018,tree_count,landcover_group
0,"POLYGON ((257332.254 672874.051, 257332.254 67...",2.0,23000,9,other
1,"POLYGON ((257492.254 672874.051, 257492.254 67...",5.0,23000,1,other
2,"POLYGON ((257502.254 672874.051, 257502.254 67...",4.0,23000,1,other
3,"POLYGON ((257582.254 672874.051, 257582.254 67...",9.0,23000,0,other
4,"POLYGON ((257592.254 672874.051, 257592.254 67...",6.0,23000,0,other


### Reclassifying landcover to urban, green, or forest 

In [28]:
clipped.dtypes

G              float64
geometry      geometry
code_2018       object
mean_noise     float64
tree_count       int32
dtype: object

In [29]:
# Convert to numeric
clipped["code_2018"] = pd.to_numeric(clipped["code_2018"], errors="coerce")

In [30]:
clipped["code_2018"].unique()

array([31000., 23000., 11210., 12220., 11100., 12210., 13400., 13300.,
       14100., 12100., 11220., 11300., 21000., 32000., 11230., 14200.,
       12230., 50000.,    nan, 12300., 13100., 11240.])

In [31]:
import geopandas as gpd
import numpy as np

# Assume your GeoDataFrame is called gdf
def classify_landcover(code):
    if code in [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]:
        return 'urban'  # Urban
    elif code in [14100, 14200, 21000, 22000, 23000, 24000]:
        return 'green'  # Green
    elif code in [25000, 31000, 32000]:
        return 'forest'  # Forest
    elif code == 50000:
        return 'water'  # Water
    else:
        return 'other'  # Or some other default

# Apply classification
clipped["landcover_group"] = clipped["code_2018"].apply(classify_landcover)

In [37]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest


In [35]:
test = clipped[clipped["landcover_group"] == "green"]
test

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.0,16,green
111857,0.957950,"POLYGON ((253705.524 658140.294, 253705.524 65...",23000.0,4.0,4,green
111858,0.955754,"POLYGON ((253744.844 658140.294, 253744.844 65...",23000.0,4.0,30,green
112024,0.959383,"POLYGON ((253626.884 658061.653, 253666.204 65...",23000.0,4.0,6,green
111941,0.961163,"POLYGON ((253626.884 658100.973, 253626.884 65...",23000.0,4.0,0,green
...,...,...,...,...,...,...
15,0.932705,"POLYGON ((257912.784 673003.323, 257912.784 67...",23000.0,0.0,1,green
16,0.914104,"POLYGON ((257952.104 673003.323, 257952.104 67...",23000.0,0.0,0,green
7,0.922904,"POLYGON ((257952.104 673042.643, 257952.104 67...",23000.0,0.0,6,green
6,0.916827,"POLYGON ((257912.784 673042.643, 257912.784 67...",23000.0,0.0,0,green


In [36]:
test["landcover_group"].unique()

array(['green'], dtype=object)

#### Dropping the water category, and all others 

In [38]:
clipped = clipped[~clipped['landcover_group'].isin(['water', 'other'])]

In [39]:
clipped

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest
...,...,...,...,...,...,...
15,0.932705,"POLYGON ((257912.784 673003.323, 257912.784 67...",23000.0,0.000000,1,green
16,0.914104,"POLYGON ((257952.104 673003.323, 257952.104 67...",23000.0,0.000000,0,green
7,0.922904,"POLYGON ((257952.104 673042.643, 257952.104 67...",23000.0,0.000000,6,green
6,0.916827,"POLYGON ((257912.784 673042.643, 257912.784 67...",23000.0,0.000000,0,green


#### setting the correct ordering 

In [41]:
clipped['landcover_group'] = pd.Categorical(
    clipped['landcover_group'],
    categories=['urban', 'green', 'forest'],
    ordered=True
)
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

In [42]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest


In [ ]:
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

## Scaling Features Between 1 and 0

In [43]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
clipped[["noise_norm", "tree_count_norm"]] = scaler.fit_transform(clipped[["mean_noise", "tree_count"]])

In [46]:
clipped.describe()

,G,code_2018,mean_noise,tree_count,noise_norm,tree_count_norm
count,238642.000000,238642.000000,238248.000000,238642.000000,238248.000000,238642.000000
mean,0.772585,13529.364571,5.212250,417.866687,0.521225,0.001472
std,0.128726,4391.632813,2.390296,10496.008650,0.239030,0.036985
min,0.000000,11100.000000,0.000000,0.000000,0.000000,0.000000
25%,0.681115,11220.000000,3.587517,3.000000,0.358752,0.000011
50%,0.776145,12220.000000,5.512898,12.000000,0.551290,0.000042
75%,0.885738,13300.000000,6.934732,24.000000,0.693473,0.000085
max,0.977799,32000.000000,10.000000,283790.000000,1.000000,1.000000


In [47]:
clipped["G"].median()

0.7761445045471191

In [45]:
clipped.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [48]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group,noise_norm,tree_count_norm
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest,0.38496,0.000028
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest,0.40000,0.000053
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest,0.40000,0.000056
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green,0.40000,0.000056
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest,0.40000,0.000102


In [50]:
clipped.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_1m.shp")

C:\Users\ibk1\AppData\Local\Temp\ipykernel_18376\2400969659.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  clipped.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_1m.shp")
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'landcover_group' to 'landcover_'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tree_count_norm' to 'tree_cou_1'
  ogr_write(


# Filling in no data values 

In [52]:
import rasterio
import numpy as np

# Load the raster
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sim_1m_clipped.tif") as src:
    noise = src.read(1)
    profile = src.profile
    nodata = src.nodata or np.nan


In [53]:
# Create a mask of missing values
mask = np.isnan(noise)


In [54]:
from scipy.ndimage import generic_filter

# Function to compute mean of non-NaNs
def nanmean_filter(values):
    return np.nanmean(values)

# Apply focal mean (kernel size = 7 → 70m across if 10m resolution)
kernel_size = 7  # Adjust depending on your radius
smoothed = generic_filter(noise, nanmean_filter, size=kernel_size, mode='constant', cval=np.nan)

In [ ]:
# Correct mask
mask = (noise == nodata) if nodata is not None else np.isnan(noise)

# Smoothing kernel (e.g., 7x7 ~ 70m window)
def nanmean_filter(values):
    return np.nanmean(values)

smoothed = generic_filter(noise.astype(float), nanmean_filter, size=7, mode='constant', cval=np.nan)

# Fill missing areas
filled = np.where(mask, smoothed, noise)
profile.update(dtype=rasterio.float32, nodata=None)

In [55]:
filled = np.where(mask, smoothed, noise)

In [56]:
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sum_1m_filled.tif", "w", **profile) as dst:
    dst.write(filled, 1)

# Vegetation GWR for 50 by 50m Grid

In [50]:
import geopandas as gpd 
ndns = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\50x50_ndvi_noise_grid.geojson")

In [51]:
ndns.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,noise_majority,ndvi_mean,geometry
0,29367.0,254824.4996,669971.4048,254874.4996,669921.4048,62.0,88.0,3.000000,3.0,0.747061,"MULTIPOLYGON (((254874.5 669960.669, 254874.5 ..."
1,29368.0,254824.4996,669921.4048,254874.4996,669871.4048,63.0,88.0,3.001072,3.0,0.946363,"MULTIPOLYGON (((254824.5 669921.405, 254874.5 ..."
2,29369.0,254824.4996,669871.4048,254874.4996,669821.4048,64.0,88.0,4.897845,3.0,0.814211,"MULTIPOLYGON (((254824.5 669871.405, 254874.5 ..."
3,29370.0,254824.4996,669821.4048,254874.4996,669771.4048,65.0,88.0,4.268470,3.0,0.777601,"MULTIPOLYGON (((254824.5 669821.405, 254874.5 ..."
4,29371.0,254824.4996,669771.4048,254874.4996,669721.4048,66.0,88.0,4.690858,3.0,0.861797,"MULTIPOLYGON (((254824.5 669771.405, 254874.5 ..."


In [53]:
from rasterstats import zonal_stats
stats = zonal_stats(ndns, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\cluster analysis\lc_cls_v2_50by50grid.tif", categorical=True)

In [54]:
import pandas as pd 

In [55]:
stats_df = pd.DataFrame(stats).fillna(0)  # fill NaN with 0 where class is missing
stats_df.columns = [f'class_{int(col)}' for col in stats_df.columns]

ndns = ndns.reset_index(drop=True)  # Ensure same index
ndns = pd.concat([ndns, stats_df], axis=1)
# Total pixels (can also use sum of class columns)
ndns['total_pix'] = stats_df.sum(axis=1)

# Example: forest (class 3) percentage
ndns['forest_pct'] = ndns['class_3'] / ndns['total_pix']
ndns['grass_pct'] = ndns['class_2'] / ndns['total_pix']
ndns['urban_pct'] = ndns['class_1'] / ndns['total_pix']


In [57]:
ndns.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,noise_majority,ndvi_mean,geometry,class_2,class_1,class_3,total_pix,forest_pct,grass_pct,urban_pct
0,29367.0,254824.4996,669971.4048,254874.4996,669921.4048,62.0,88.0,3.000000,3.0,0.747061,"MULTIPOLYGON (((254874.5 669960.669, 254874.5 ...",1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,29368.0,254824.4996,669921.4048,254874.4996,669871.4048,63.0,88.0,3.001072,3.0,0.946363,"MULTIPOLYGON (((254824.5 669921.405, 254874.5 ...",1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,29369.0,254824.4996,669871.4048,254874.4996,669821.4048,64.0,88.0,4.897845,3.0,0.814211,"MULTIPOLYGON (((254824.5 669871.405, 254874.5 ...",0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,29370.0,254824.4996,669821.4048,254874.4996,669771.4048,65.0,88.0,4.268470,3.0,0.777601,"MULTIPOLYGON (((254824.5 669821.405, 254874.5 ...",0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,29371.0,254824.4996,669771.4048,254874.4996,669721.4048,66.0,88.0,4.690858,3.0,0.861797,"MULTIPOLYGON (((254824.5 669771.405, 254874.5 ...",0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [58]:
ndns.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\cluster analysis\cluster_input_gg.gpkg", driver="GPKG")

In [69]:
stats = zonal_stats(ndns, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\lc_class_rast_gg.tif", categorical=True)

# Each dict in `stats` will have keys like: {1: count, 2: count, 3: count}
# Add them to your GeoDataFrame
ndns["urban_count"] = [s.get(1, 0) for s in stats]
ndns["green_count"] = [s.get(2, 0) for s in stats]
ndns["forest_count"] = [s.get(3, 0) for s in stats]

In [70]:
ndns["total"] = ndns[["urban_count", "green_count", "forest_count"]].sum(axis=1)
ndns["urban_share"] = ndns["urban_count"] / ndns["total"]

In [72]:
ndns.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,noise_majority,ndvi_mean,geometry,urban_count,green_count,forest_count,total,urban_share
0,29367.0,254824.4996,669971.4048,254874.4996,669921.4048,62.0,88.0,3.000000,3.0,0.747061,"MULTIPOLYGON (((254874.5 669960.669, 254874.5 ...",0,1,0,1,0.0
1,29368.0,254824.4996,669921.4048,254874.4996,669871.4048,63.0,88.0,3.001072,3.0,0.946363,"MULTIPOLYGON (((254824.5 669921.405, 254874.5 ...",1,0,0,1,1.0
2,29369.0,254824.4996,669871.4048,254874.4996,669821.4048,64.0,88.0,4.897845,3.0,0.814211,"MULTIPOLYGON (((254824.5 669871.405, 254874.5 ...",1,0,0,1,1.0
3,29370.0,254824.4996,669821.4048,254874.4996,669771.4048,65.0,88.0,4.268470,3.0,0.777601,"MULTIPOLYGON (((254824.5 669821.405, 254874.5 ...",1,0,0,1,1.0
4,29371.0,254824.4996,669771.4048,254874.4996,669721.4048,66.0,88.0,4.690858,3.0,0.861797,"MULTIPOLYGON (((254824.5 669771.405, 254874.5 ...",1,0,0,1,1.0


In [74]:
from geopandas.tools import sjoin

# Perform the spatial join only once, with how="left"
joined = sjoin(ndns, clip_trees, how="left")

# Now group by the index of the left GeoDataFrame (ndns)
tree_counts = joined.groupby(joined.index).size()

# Add counts to the original GeoDataFrame
ndns["tree_count"] = ndns.index.map(tree_counts).fillna(0).astype(int)


In [75]:
ndns.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,noise_majority,ndvi_mean,geometry,urban_count,green_count,forest_count,total,urban_share,tree_count
0,29367.0,254824.4996,669971.4048,254874.4996,669921.4048,62.0,88.0,3.000000,3.0,0.747061,"MULTIPOLYGON (((254874.5 669960.669, 254874.5 ...",0,1,0,1,0.0,20
1,29368.0,254824.4996,669921.4048,254874.4996,669871.4048,63.0,88.0,3.001072,3.0,0.946363,"MULTIPOLYGON (((254824.5 669921.405, 254874.5 ...",1,0,0,1,1.0,19
2,29369.0,254824.4996,669871.4048,254874.4996,669821.4048,64.0,88.0,4.897845,3.0,0.814211,"MULTIPOLYGON (((254824.5 669871.405, 254874.5 ...",1,0,0,1,1.0,7
3,29370.0,254824.4996,669821.4048,254874.4996,669771.4048,65.0,88.0,4.268470,3.0,0.777601,"MULTIPOLYGON (((254824.5 669821.405, 254874.5 ...",1,0,0,1,1.0,18
4,29371.0,254824.4996,669771.4048,254874.4996,669721.4048,66.0,88.0,4.690858,3.0,0.861797,"MULTIPOLYGON (((254824.5 669771.405, 254874.5 ...",1,0,0,1,1.0,17


In [76]:
ggbounds.explore()

In [77]:
ndns["x"] = ndns.geometry.centroid.x
ndns["y"] = ndns.geometry.centroid.y

In [79]:
ndns.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,noise_majority,ndvi_mean,geometry,urban_count,green_count,forest_count,total,urban_share,tree_count,x,y
0,29367.0,254824.4996,669971.4048,254874.4996,669921.4048,62.0,88.0,3.000000,3.0,0.747061,"MULTIPOLYGON (((254874.5 669960.669, 254874.5 ...",0,1,0,1,0.0,20,254850.529389,669938.966456
1,29368.0,254824.4996,669921.4048,254874.4996,669871.4048,63.0,88.0,3.001072,3.0,0.946363,"MULTIPOLYGON (((254824.5 669921.405, 254874.5 ...",1,0,0,1,1.0,19,254849.499600,669896.404800
2,29369.0,254824.4996,669871.4048,254874.4996,669821.4048,64.0,88.0,4.897845,3.0,0.814211,"MULTIPOLYGON (((254824.5 669871.405, 254874.5 ...",1,0,0,1,1.0,7,254849.499600,669846.404800
3,29370.0,254824.4996,669821.4048,254874.4996,669771.4048,65.0,88.0,4.268470,3.0,0.777601,"MULTIPOLYGON (((254824.5 669821.405, 254874.5 ...",1,0,0,1,1.0,18,254849.499600,669796.404800
4,29371.0,254824.4996,669771.4048,254874.4996,669721.4048,66.0,88.0,4.690858,3.0,0.861797,"MULTIPOLYGON (((254824.5 669771.405, 254874.5 ...",1,0,0,1,1.0,17,254849.499600,669746.404800


In [83]:
ndns[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_gwr_input_50m.shp", index=False)

C:\Users\ibk1\AppData\Local\Temp\ipykernel_23224\3095915351.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  ndns[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_gwr_input_50m.shp", index=False)
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'urban_count' to 'urban_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'green_count' to 'green_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'forest_count' to 'forest_cou'
  ogr_write(


In [84]:
len(ndns)

71881

# GWR Preprocessing by Datazones

In [88]:
# Reading the datazones shapefile 
ggdz = gpd.read_file(r"C:/Users/ibk1/NoiseModelling/Glasgow/env_data/gg_te/gg_te.shp")
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_1m_filled.tif", categorical=False)
ggdz["mean_noise"] = [s["mean"] for s in stats]

In [90]:
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_1m_filled.tif", categorical=False)

In [91]:
ggdz["mean_noise"] = [s["mean"] for s in stats]

In [92]:
ggdz.columns

Index(['bge_code', 'bge_type', 'la_name', 'co_name', 'reg_name', 'country',
       'total_pop', 'urban_area', 'urban_pct', 'tc_goal', 'treecanopy',
       'tc_gap', 'priority_i', 'inc_rank', 'incnorm', 'inc_dec', 'emp_rank',
       'empnorm', 'emp_dec', 'hlth_rank', 'hlthnorm', 'hlth_dec', 'temp_diff',
       'tempnorm', 'NO2_avg', 'PM25_avg', 'apb_index', 'dep_ratio',
       'depratnorm', 'dep_perc', 'tes', 'la_tes', 'pctmineth', 'pct_child',
       'pct_senior', 'la_code', 'co_code', 'peat', 'geometry', 'mean_noise'],
      dtype='object')

In [36]:
ggdz = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_GWR_Input_summer_datazones.shp") 

In [95]:
ggdz.explore(column = "apb_index")

In [ ]:
ggdz.explore(column = "apb_index")

In [37]:
SIMD = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\SIMD Data\SG_SIMD_2020.shp")

In [38]:
SIMD.head()

,DataZone,DZName,LAName,SAPE2017,WAPE2017,Rankv2,Quintilev2,Decilev2,Vigintilv2,Percentv2,...,CrimeRate,CrimeRank,HouseNumOC,HouseNumNC,HouseOCrat,HouseNCrat,HouseRank,Shape_Leng,Shape_Area,geometry
0,S01006506,Culter - 01,Aberdeen City,894,580,4691,4,7,14,68,...,125,4664.0,87,10,10%,1%,3248.0,11801.872345,4.388802e+06,"POLYGON ((383285.265 800510.607, 383348.492 80..."
1,S01006507,Culter - 02,Aberdeen City,793,470,4862,4,7,14,70,...,128,4602.0,85,4,10%,0%,3486.0,2900.406362,2.217468e+05,"POLYGON ((383527.919 801536.276, 383541.089 80..."
2,S01006508,Culter - 03,Aberdeen City,624,461,5686,5,9,17,82,...,130,4563.5,31,8,5%,1%,5342.0,3468.761949,2.701948e+05,"POLYGON ((383473 801227, 383597 801087, 383598..."
3,S01006509,Culter - 04,Aberdeen City,537,307,4332,4,7,13,63,...,75,5626.0,42,6,7%,1%,4394.5,1647.461389,9.625426e+04,"POLYGON ((383976.659 801182.579, 383984.102 80..."
4,S01006510,Culter - 05,Aberdeen City,663,415,3913,3,6,12,57,...,168,3885.0,50,7,9%,1%,3736.0,3026.111412,1.800766e+05,"POLYGON ((384339 801211, 384316.51 801182.159,..."


In [39]:
SIMD.columns

Index(['DataZone', 'DZName', 'LAName', 'SAPE2017', 'WAPE2017', 'Rankv2',
       'Quintilev2', 'Decilev2', 'Vigintilv2', 'Percentv2', 'IncRate',
       'IncNumDep', 'IncRankv2', 'EmpRate', 'EmpNumDep', 'EmpRank', 'HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank', 'EduAttend', 'EduAttain', 'EduNoQuals',
       'EduPartici', 'EduUniver', 'EduRank', 'GAccPetrol', 'GAccDTGP',
       'GAccDTPost', 'GAccDTPsch', 'GAccDTSsch', 'GAccDTRet', 'GAccPTGP',
       'GAccPTPost', 'GAccPTRet', 'GAccBrdbnd', 'GAccRank', 'CrimeCount',
       'CrimeRate', 'CrimeRank', 'HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', 'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')

In [40]:
ggsimd = SIMD[SIMD["LAName"] == "Glasgow City"]

In [106]:
ggdz.head(1)

,bge_code,bge_type,la_name,co_name,reg_name,country,total_pop,urban_area,urban_pct,tc_goal,...,tes,la_tes,pctmineth,pct_child,pct_senior,la_code,co_code,peat,geometry,mean_noise
0,S01009758,Data Zone,Glasgow City,Glasgow South West,South Scotland,Scotland,598,0.077301,0.99,0.24,...,81,80.0,0.073579,0.173913,0.172241,S12000049,S14000035,0,"POLYGON ((254445.813 658652.083, 254381.766 65...",4.351854


In [111]:
ggsimd.columns

Index(['DataZone', 'DZName', 'LAName', 'SAPE2017', 'WAPE2017', 'Rankv2',
       'Quintilev2', 'Decilev2', 'Vigintilv2', 'Percentv2', 'IncRate',
       'IncNumDep', 'IncRankv2', 'EmpRate', 'EmpNumDep', 'EmpRank', 'HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank', 'EduAttend', 'EduAttain', 'EduNoQuals',
       'EduPartici', 'EduUniver', 'EduRank', 'GAccPetrol', 'GAccDTGP',
       'GAccDTPost', 'GAccDTPsch', 'GAccDTSsch', 'GAccDTRet', 'GAccPTGP',
       'GAccPTPost', 'GAccPTRet', 'GAccBrdbnd', 'GAccRank', 'CrimeCount',
       'CrimeRate', 'CrimeRank', 'HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', 'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')

In [45]:
cols_to_keep = ['DataZone','HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank','HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', "Vigintilv2",'EduRank', "GAccRank", "CrimeRate"
               ]
merged = ggdz.merge(ggsimd[cols_to_keep], left_on="bge_code", right_on="DataZone", how="left")


In [46]:
merged.explore(column = "HlthDprsPc")

In [47]:
# Remove % and convert to float
merged["HouseOCrat"] = merged["HouseOCrat"].str.replace("%", "").astype(float)
merged["HouseNCrat"] = merged["HouseNCrat"].str.replace("%", "").astype(float)
merged["HlthDprsPc"] = merged["HlthDprsPc"].str.replace("%", "").astype(float)
merged["HlthLBWTPc"] = merged["HlthLBWTPc"].str.replace("%", "").astype(float)
# Optional: Convert to decimals (0.1 instead of 10)
# ggsimd["HouseOCrat"] /= 100
# ggsimd["HouseNCrat"] /= 100

In [48]:
merged.columns

Index(['bge_code', 'bge_type', 'la_name', 'co_name', 'reg_name', 'country',
       'total_pop', 'urban_area', 'urban_pct', 'tc_goal', 'treecanopy',
       'tc_gap', 'priority_i', 'inc_rank', 'incnorm', 'inc_dec', 'emp_rank',
       'empnorm', 'emp_dec', 'hlth_rank', 'hlthnorm', 'hlth_dec', 'temp_diff',
       'tempnorm', 'NO2_avg', 'PM25_avg', 'apb_index', 'dep_ratio',
       'depratnorm', 'dep_perc', 'tes', 'la_tes', 'pctmineth', 'pct_child',
       'pct_senior', 'la_code', 'co_code', 'peat', 'mean_noise', 'geometry',
       'DataZone', 'HlthCIF', 'HlthAlcSR', 'HlthDrugSR', 'HlthSMR',
       'HlthDprsPc', 'HlthLBWTPc', 'HlthEmergS', 'HlthRank', 'HouseNumOC',
       'HouseNumNC', 'HouseOCrat', 'HouseNCrat', 'HouseRank', 'Vigintilv2',
       'EduRank', 'GAccRank', 'CrimeRate'],
      dtype='object')

In [49]:
merged.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_GWR_Input_summer_datazones_SIMD_extra.shp") 

## Winter 

In [130]:
# Reading the datazones shapefile 
ggdz_wint = gpd.read_file(r"C:/Users/ibk1/NoiseModelling/Glasgow/env_data/gg_te/gg_te.shp")
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\winter\gg_winter_1m_raster.tif", categorical=False)
ggdz_wint["mean_noise"] = [s["mean"] for s in stats]

In [132]:
ggdz_wint.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\winter\gg_GWR_Input_WINTER_datazones.shp") 